# Phase 1: Tox21 Data Cleaning & Normalization
This notebook perfectly implements the Phase 1 pipeline from the project roadmap, addressing salt-stripping, tautomer normalization, stereochemistry preservation, and 'toxic-overrules-safe' deduplication for the Tox21 dataset.

In [17]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem.MolStandardize import rdMolStandardize

print("✅ Libraries loaded successfully.")

✅ Libraries loaded successfully.


In [18]:
# All 12 Tox21 target endpoints
TARGET_COLS = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 
    'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53'
]

# Initialize RDKit normalizers
remover = SaltRemover()
te = rdMolStandardize.TautomerEnumerator()
te.SetMaxTautomers(50)

failed_smiles = []

def normalize_molecule(smiles: str) -> str:
    """Validate → Strip salts → Normalize tautomer. Preserves stereochemistry."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            failed_smiles.append((smiles, 'parse_failure'))
            return None
        
        # Strip counterions (e.g., Na+, Cl-) but keep main fragment
        mol = remover.StripMol(mol, dontRemoveEverything=True)
        
        # Standardize tautomer
        canonical_mol = te.Canonicalize(mol)
        
        # Return canonical SMILES keeping stereochemistry (isomericSmiles=True)
        return Chem.MolToSmiles(canonical_mol, isomericSmiles=True)
        
    except Exception as e:
        failed_smiles.append((smiles, str(e)))
        return None

In [19]:
def deduplicate_toxic_overrules(df: pd.DataFrame) -> pd.DataFrame:
    """
    If the same molecule appears twice with conflicting labels across any endpoints:
    Toxic (1) overrules Safe (0). Valid labels (0/1) overrule Missing (-1).
    We use groupby.agg('max') to mathematically guarantee this.
    """
    print(f"Dataset size before deduplication: {len(df)}")
    
    # Temporarily fill missing values with -1 for the math to work
    df[TARGET_COLS] = df[TARGET_COLS].fillna(-1)
    
    # Toxic (1) > Safe (0) > Missing (-1)
    agg_dict = {col: 'max' for col in TARGET_COLS}
    agg_dict['mol_id'] = 'first' # keep the first ID found
    
    df_dedup = df.groupby('clean_smiles').agg(agg_dict).reset_index()
    
    # CRITICAL: convert -1 sentinel back to NaN so models don't learn -1 as a class
    df_dedup[TARGET_COLS] = df_dedup[TARGET_COLS].replace(-1, np.nan)
    
    print(f"Dataset size after deduplication: {len(df_dedup)}")
    return df_dedup

In [20]:
print("Loading raw Tox21 data...")
df = pd.read_csv('../data/raw/tox21.csv')
display(df.head())

Loading raw Tox21 data...


,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O


In [21]:
print("Normalizing molecules (salt stripping & tautomer standardization)...")
# Using apply instead of progress_apply to guarantee stability
df['clean_smiles'] = df['smiles'].apply(normalize_molecule)

invalid_count = df['clean_smiles'].isna().sum()
if invalid_count > 0:
    print(f"Dropping {invalid_count} invalid molecules.")
    df = df.dropna(subset=['clean_smiles'])

Normalizing molecules (salt stripping & tautomer standardization)...


[10:38:04] WARNING: not removing hydrogen atom without neighbors
[10:38:05] Tautomer enumeration stopped at 50 tautomers: max tautomers reached
[10:38:05] Tautomer enumeration stopped at 50 tautomers: max tautomers reached
[10:38:05] Tautomer enumeration stopped at 50 tautomers: max tautomers reached
[10:38:05] Tautomer enumeration stopped at 50 tautomers: max tautomers reached
[10:38:05] Tautomer enumeration stopped at 50 tautomers: max tautomers reached
[10:38:05] Tautomer enumeration stopped at 50 tautomers: max tautomers reached
[10:38:05] Tautomer enumeration stopped at 50 tautomers: max tautomers reached
[10:38:06] Tautomer enumeration stopped at 50 tautomers: max tautomers reached
[10:38:06] Tautomer enumeration stopped at 50 tautomers: max tautomers reached
[10:38:06] Tautomer enumeration stopped at 50 tautomers: max tautomers reached
[10:38:07] Tautomer enumeration stopped at 50 tautomers: max tautomers reached
[10:38:07] Tautomer enumeration stopped at 50 tautomers: max tauto

Dropping 1 invalid molecules.


[10:38:36] Tautomer enumeration stopped at 50 tautomers: max tautomers reached
[10:38:36] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 7 11
[10:38:36] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 7 11


In [22]:
print("Applying 'Toxic-Overrules-Safe' deduplication...")
df_clean = deduplicate_toxic_overrules(df)

# Reorder columns for readability
cols = ['mol_id', 'clean_smiles'] + TARGET_COLS
df_clean = df_clean[cols]
display(df_clean.head())

Applying 'Toxic-Overrules-Safe' deduplication...
Dataset size before deduplication: 7830
Dataset size after deduplication: 7800


,mol_id,clean_smiles,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53
0,TOX1374,BrC(Br)Br,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,TOX5830,BrC(Br)C(Br)(Br)Br,0.0,0.0,NaN,NaN,0.0,0.0,0.0,NaN,0.0,NaN,0.0,1.0
2,TOX6083,BrC(Br)C(Br)Br,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,TOX4941,BrC/C=C/CBr,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0
4,TOX7527,BrC1CCC(Br)C(Br)CCC(Br)C(Br)CCC1Br,0.0,0.0,0.0,NaN,NaN,0.0,0.0,1.0,0.0,NaN,1.0,NaN


In [23]:
output_path = '../data/processed/tox21_cleaned.csv'
print(f"Saving cleaned dataset to {output_path}...")
df_clean.to_csv(output_path, index=False)
print("\u2705 Phase 1 Data Cleaning Complete.")

Saving cleaned dataset to ../data/processed/tox21_cleaned.csv...
✅ Phase 1 Data Cleaning Complete.


In [24]:
if len(failed_smiles) > 0:
    failed_path = '../data/processed/failed_smiles_log.csv'
    print(f"Saving log of {len(failed_smiles)} failed molecules to {failed_path}...")
    pd.DataFrame(failed_smiles, columns=['smiles','reason']).to_csv(failed_path, index=False)
else:
    print("\u2705 No SMILES parsing failures to log.")

Saving log of 1 failed molecules to ../data/processed/failed_smiles_log.csv...
